# 08 — Basic Evaluation

In this notebook we will:
1. Understand why RAG evaluation matters
2. Create a small test set of question/answer pairs
3. Measure **retrieval quality** — does the retriever find the right chunks?
4. Measure **answer quality** — is the LLM's answer faithful to the context?
5. Identify failure modes and areas for improvement

### Why evaluate?
Without evaluation, you're flying blind. A RAG system can fail in two places:
- **Retrieval failure**: The right chunks aren't found → the LLM can't answer correctly no matter how good it is.
- **Generation failure**: The right chunks are found, but the LLM ignores them, summarizes poorly, or hallucinates.

We'll measure both separately.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv("../.env")

# ChromaDB
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
)

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print(f"Collection: {collection.count()} documents")

In [ ]:
# Our RAG pipeline (reused from notebook 04)
def retrieve(query, n_results=5, where=None):
    results = collection.query(
        query_texts=[query], n_results=n_results, where=where,
    )
    context_parts = []
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        context_parts.append(
            f"[{meta.get('speaker', '?')} — {meta.get('role', '?')} | {meta.get('quarter', '?')}]\n{doc}"
        )
    return results, "\n\n---\n\n".join(context_parts)

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an analyst assistant answering questions about earnings calls.
Use ONLY the provided context. If the answer is not in the context, say so.
Cite the speaker and quarter when relevant.

Context:
{context}

Question: {question}
""")

chain = RAG_PROMPT | llm | StrOutputParser()

def ask(question, n_results=5, where=None):
    results, context = retrieve(question, n_results, where)
    answer = chain.invoke({"context": context, "question": question})
    return results, context, answer

print("RAG pipeline ready")

## Step 1: Create the test set

---

Create a list of test cases. Each case has:
- `question`: the query
- `expected_quarter`: which quarter(s) should appear in results
- `expected_speaker`: who should be in the results (if relevant)
- `expected_keywords`: key terms that MUST appear in a correct answer
- `category`: type of question (factual, speaker-specific, temporal, off-topic)

Create at least 8 test cases covering different categories. You know the transcripts — use real data points you've seen in the notebooks.

In [ ]:
test_cases = [
    # Factual questions (specific numbers)
    {
        "question": "What were Apple's total revenue results in Q4 2025?",
        "expected_quarter": ["Q4-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["102.5", "billion", "8%"],
        "category": "factual",
    },
    {
        "question": "What was Apple's revenue in Q3 2025?",
        "expected_quarter": ["Q3-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["94", "billion", "10%"],
        "category": "factual",
    },
    # Speaker-specific questions
    {
        "question": "What did the CFO say about gross margins in Q4 2025?",
        "expected_quarter": ["Q4-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["margin", "gross"],
        "category": "speaker-specific",
    },
    {
        "question": "What did Tim Cook say about Apple Intelligence?",
        "expected_quarter": ["Q3-2025", "Q4-2025", "Q1-2026"],
        "expected_speaker": "Timothy D. Cook",
        "expected_keywords": ["Apple Intelligence"],
        "category": "speaker-specific",
    },
    # Quarter-specific questions
    {
        "question": "What were the iPhone results in Q3 2025?",
        "expected_quarter": ["Q3-2025"],
        "expected_speaker": "",
        "expected_keywords": ["iPhone"],
        "category": "quarter-specific",
    },
    {
        "question": "What is Apple's revenue guidance for the next quarter in Q1 2026?",
        "expected_quarter": ["Q1-2026"],
        "expected_speaker": "",
        "expected_keywords": ["guidance", "expect"],
        "category": "quarter-specific",
    },
    # Cross-quarter question
    {
        "question": "How has Apple's revenue changed across quarters?",
        "expected_quarter": ["Q3-2025", "Q4-2025"],
        "expected_speaker": "",
        "expected_keywords": ["94", "102.5"],
        "category": "cross-quarter",
    },
    # Off-topic question
    {
        "question": "What is Apple's strategy for electric vehicles?",
        "expected_quarter": [],
        "expected_speaker": "",
        "expected_keywords": [],
        "category": "off-topic",
    },
]

print(f"Test cases: {len(test_cases)}")
for tc in test_cases:
    print(f"  [{tc['category']}] {tc['question'][:70]}")

## Step 2: Evaluate retrieval quality

For each test case, we check:
- Does the retriever return chunks from the expected quarter?
- Does the retriever return chunks from the expected speaker?
- Do the retrieved chunks contain the expected keywords?

In [ ]:
def evaluate_retrieval(test_cases):
    """Evaluate retrieval quality for each test case."""
    results_log = []
    
    for tc in test_cases:
        raw_results, context = retrieve(tc["question"])
        
        # Check quarter coverage
        retrieved_quarters = [raw_results["metadatas"][0][i]["quarter"] 
                             for i in range(len(raw_results["documents"][0]))]
        expected_q = tc.get("expected_quarter", [])
        quarter_hit = any(q in retrieved_quarters for q in expected_q) if expected_q else True
        
        # Check speaker coverage
        retrieved_speakers = [raw_results["metadatas"][0][i]["speaker"] 
                             for i in range(len(raw_results["documents"][0]))]
        expected_s = tc.get("expected_speaker", "")
        speaker_hit = expected_s in retrieved_speakers if expected_s else True
        
        # Check keyword coverage in retrieved context
        context_lower = context.lower()
        keyword_hits = [kw for kw in tc["expected_keywords"] if kw.lower() in context_lower]
        keyword_score = len(keyword_hits) / len(tc["expected_keywords"]) if tc["expected_keywords"] else 1.0
        
        results_log.append({
            "question": tc["question"],
            "category": tc["category"],
            "quarter_hit": quarter_hit,
            "speaker_hit": speaker_hit,
            "keyword_score": keyword_score,
            "keyword_hits": keyword_hits,
            "keyword_misses": [kw for kw in tc["expected_keywords"] if kw.lower() not in context_lower],
            "retrieved_quarters": retrieved_quarters,
        })
    
    return results_log

retrieval_results = evaluate_retrieval(test_cases)

# Summary
print("RETRIEVAL EVALUATION")
print("=" * 70)
for r in retrieval_results:
    status = "PASS" if r["quarter_hit"] and r["speaker_hit"] and r["keyword_score"] >= 0.5 else "FAIL"
    print(f"\n[{status}] [{r['category']}] {r['question'][:60]}")
    print(f"  Quarter hit: {r['quarter_hit']} | Speaker hit: {r['speaker_hit']} | Keyword score: {r['keyword_score']:.0%}")
    if r["keyword_misses"]:
        print(f"  Missing keywords: {r['keyword_misses']}")

# Overall metrics
total = len(retrieval_results)
quarter_hits = sum(1 for r in retrieval_results if r["quarter_hit"])
speaker_hits = sum(1 for r in retrieval_results if r["speaker_hit"])
avg_keyword = sum(r["keyword_score"] for r in retrieval_results) / total

print(f"\n{'=' * 70}")
print(f"Quarter accuracy: {quarter_hits}/{total} ({quarter_hits/total:.0%})")
print(f"Speaker accuracy: {speaker_hits}/{total} ({speaker_hits/total:.0%})")
print(f"Avg keyword coverage: {avg_keyword:.0%}")

## Step 3: Evaluate answer quality

Now we check the LLM's generated answers. We use the LLM itself as a judge (a common technique called **LLM-as-judge**).

In [ ]:
JUDGE_PROMPT = ChatPromptTemplate.from_template("""
You are evaluating a RAG system's answer. Score it on two dimensions:

1. **Faithfulness** (1-5): Does the answer only use information from the context? 
   5 = fully grounded, 1 = clearly hallucinated.

2. **Relevance** (1-5): Does the answer address the question?
   5 = directly answers, 1 = completely off-topic.

Also check if these keywords appear in the answer: {expected_keywords}

Context provided to the system:
{context}

Question: {question}
Answer: {answer}

Respond in this exact format:
Faithfulness: [1-5]
Relevance: [1-5]
Keywords found: [list]
Brief explanation: [one sentence]
""")

judge_chain = JUDGE_PROMPT | llm | StrOutputParser()
print("Judge chain ready")

In [ ]:
import re

def evaluate_answers(test_cases):
    """Run the full RAG pipeline and judge each answer."""
    results_log = []
    
    for tc in test_cases:
        print(f"  Evaluating: {tc['question'][:50]}...", end=" ")
        
        # Get RAG answer
        raw_results, context, answer = ask(tc["question"])
        
        # Judge it
        judgment = judge_chain.invoke({
            "context": context,
            "question": tc["question"],
            "answer": answer,
            "expected_keywords": ", ".join(tc["expected_keywords"]),
        })
        
        # Parse scores
        faithfulness = int(re.search(r"Faithfulness:\s*(\d)", judgment).group(1)) if re.search(r"Faithfulness:\s*(\d)", judgment) else 0
        relevance = int(re.search(r"Relevance:\s*(\d)", judgment).group(1)) if re.search(r"Relevance:\s*(\d)", judgment) else 0
        
        results_log.append({
            "question": tc["question"],
            "category": tc["category"],
            "answer": answer,
            "faithfulness": faithfulness,
            "relevance": relevance,
            "judgment": judgment,
        })
        print(f"F:{faithfulness}/5 R:{relevance}/5")
    
    return results_log

print("Running evaluation...\n")
answer_results = evaluate_answers(test_cases)

In [ ]:
# Detailed results
print("ANSWER EVALUATION")
print("=" * 70)
for r in answer_results:
    print(f"\n[{r['category']}] {r['question'][:60]}")
    print(f"  Faithfulness: {r['faithfulness']}/5 | Relevance: {r['relevance']}/5")
    print(f"  Answer: {r['answer'][:150]}...")

# Overall metrics
total = len(answer_results)
avg_faith = sum(r["faithfulness"] for r in answer_results) / total
avg_rel = sum(r["relevance"] for r in answer_results) / total

print(f"\n{'=' * 70}")
print(f"Avg Faithfulness: {avg_faith:.1f}/5")
print(f"Avg Relevance: {avg_rel:.1f}/5")

# Per-category breakdown
print(f"\nPer category:")
categories = set(r["category"] for r in answer_results)
for cat in sorted(categories):
    cat_results = [r for r in answer_results if r["category"] == cat]
    cat_faith = sum(r["faithfulness"] for r in cat_results) / len(cat_results)
    cat_rel = sum(r["relevance"] for r in cat_results) / len(cat_results)
    print(f"  {cat}: F={cat_faith:.1f} R={cat_rel:.1f} (n={len(cat_results)})")

## Step 4: Identify failure modes

Let's look at the worst-performing test cases to understand where the system breaks.

In [ ]:
# Find the weakest answers
sorted_results = sorted(answer_results, key=lambda r: r["faithfulness"] + r["relevance"])

print("WEAKEST ANSWERS (bottom 3):")
print("=" * 70)
for r in sorted_results[:3]:
    print(f"\n[{r['category']}] {r['question']}")
    print(f"  Scores: F={r['faithfulness']}/5 R={r['relevance']}/5")
    print(f"  Answer: {r['answer'][:200]}")
    print(f"  Judge: {r['judgment']}")

## Summary

In this notebook you learned:
- RAG evaluation has two dimensions: **retrieval quality** and **answer quality**
- Retrieval is measured by whether the right chunks are found (quarter, speaker, keywords)
- Answer quality is measured by faithfulness (grounding) and relevance
- **LLM-as-judge** is a common technique to evaluate generated answers at scale
- Identifying failure modes tells you where to invest: better chunking? better prompts? better retrieval?

### Phase 2 complete!

You now have a multi-document RAG system with:
- Multiple companies and quarters
- Automatic metadata filtering (SelfQueryRetriever)
- Temporal comparison across quarters
- Basic evaluation framework

**Next phase:** Migration to AWS (S3, Bedrock, OpenSearch).